## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:

%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [3]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [6]:
# from ner.dataset_base import HuggingFaceMultilingualDataset
# class Wikiann(HuggingFaceMultilingualDataset):
#     dataset_name = 'wikiann'
#     language = 'ro'
#     license = 'unknown'

# dataset = Wikiann()
# dataset.check_labels()

In [7]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='hi')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/1000 [00:00<?, ?it/s]

In [8]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-LOC', 'I-ORG', 'B-PER', 'I-LOC', 'B-ORG', 'I-PER', 'O'}


## naamapadam

In [9]:
label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

naamapadam = ner.ReadNERData()
naamapadam_words, naamapadam_labels = naamapadam.read_dataset('ai4bharat/naamapadam', label_map, lang='hi')

Generating train split:   0%|          | 0/985787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/867 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13460 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/867 [00:00<?, ?it/s]

# Evaluate model

In [ ]:
alignment = {
'B-organization': 'B-ORG',
'O': 'O',
'B-other': 'O',
'B-person': 'B-PER',
'I-person': 'I-PER',
'B-location': 'B-LOC',
'I-organization': 'I-ORG',
'I-other': 'O',
'I-location': 'I-LOC'
}

model_name = "ai4bharat/IndicNER"
model_name_output = 'ai4bharat/IndicNER'
model_evaluation = ner.ModelEvaluation(
    model_name,
    # alignment
)

tokenizer_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/667M [00:00<?, ?B/s]

In [ ]:
model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-ORG',
 2: 'B-PER',
 3: 'I-LOC',
 4: 'I-ORG',
 5: 'I-PER',
 6: 'O'}

### wikiann

In [ ]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

In [ ]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

In [ ]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

### naamapadam

In [ ]:
data_name = "naamapadam"
naamapadam_evaluation_output = model_evaluation.evaluate_model(naamapadam_words, naamapadam_labels)

  0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
naamapadam_seqeval = naamapadam_evaluation_output.get_classification('Seqeval')
naamapadam_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6876,0.9450,0.7960,1728
1,ORG,0.0006,0.0027,0.0010,373
2,PER,0.6818,0.2908,0.4077,4230
3,micro,0.4867,0.4524,0.4689,6331
4,macro,0.4567,0.4128,0.4016,6331
5,weighted,0.6433,0.4524,0.4897,6331


In [ ]:
naamapadam_sklearn = naamapadam_evaluation_output.get_classification('Sklearn')
naamapadam_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7166,0.9554,0.8189,1728
1,B-ORG,0.0013,0.0054,0.0020,373
2,B-PER,0.7503,0.3161,0.4448,4230
3,I-LOC,0.3781,0.8462,0.5226,273
4,I-ORG,0.0042,0.0179,0.0068,503
5,I-PER,0.9672,0.5526,0.7033,2186
6,O,0.9517,0.9470,0.9494,79176
7,accuracy,0.8977,88469,None,None
8,macro,0.5385,0.5201,0.4926,88469
9,weighted,0.9267,0.8977,0.9060,88469
